> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 9 · Notebook 06 — Factor models and cross-sectional premia

**Sessions:** S6 (Factor models & cross-sectional strategies) · [Lesson plan](../../docs/lessons/PART_09_STATISTICAL_TRADING.md) · graded labs in [`labs/part09/`](../../labs/part09/)

**You will:**
1. Compute a Newey–West standard error for an autocorrelated series.
2. Estimate factor premia with Fama–MacBeth regressions and recover a planted premium.
3. See how many months a premium needs before it is significant.
4. Neutralize a signal against sectors and beta.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with known parameters (true half-lives, true pairs, true hedge ratios, planted premia), so every estimator can be checked against the truth.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p9lib.py is in notebooks/part09/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p9lib as p

p.use_course_style()
import statsmodels.api as sm

## 1. Standard errors of autocorrelated means

`std/√n` assumes independent observations. When the series is persistent (overlapping returns, strategy P&L, monthly slopes), neighbouring errors don't cancel and the true uncertainty is larger. **Newey–West** adds the autocovariances `γ_j = (1/n) Σ (x_t − x̄)(x_{t−j} − x̄)` with declining (Bartlett) weights:

`se = √[(γ0 + 2 Σ_{j=1..L} (1 − j/(L+1)) γ_j) / n]`

The code computes `γ0`; add the lag terms (`d[j:] @ d[:-j] / n` is `γ_j`).

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def newey_west_se(x, lags):
    x = np.asarray(x, dtype=float)
    n, d = x.size, x - x.mean()
    s = d @ d / n                                         # γ0
    for j in range(1, lags + 1):
        s += 2 * (1 - j / (lags + 1)) * (d[j:] @ d[:-j]) / n
    return float(np.sqrt(s / n))

x = p.simulate_ou(500, np.log(2) / 5, 0.001, 0.01, seed=2)       # a persistent series (half-life 5)
iid = np.random.default_rng(3).normal(0.001, 0.01, 500)
cases = [(x, 0), (x, 5), (x, 10), (iid, 5)]
mine = [p.attempt(newey_west_se, *c) for c in cases]
mine = p.check("newey_west_se", mine, [p.newey_west_se(*c) for c in cases])
hac = sm.OLS(x, np.ones(x.size)).fit(cov_type="HAC", cov_kwds={"maxlags": 10, "use_correction": False}).bse[0]
pd.Series({"naive std/√n": x.std(ddof=1) / np.sqrt(x.size), "Newey–West, 5 lags": mine[1], "Newey–West, 10 lags": mine[2],
           "statsmodels HAC, 10 lags": hac}, name="standard error of the mean").to_frame()

For this persistent series the honest standard error is about 2.5 times the naive one, so a naive t-statistic would be 2.5 times too large. Your function matches `statsmodels`' HAC estimator.

## 2. Fama–MacBeth

Each month `t`, regress the **next** month's returns across stocks on the characteristics known at `t` (with a constant). The average slope is the premium; its t-statistic uses the standard deviation of the monthly slopes (`/√n`), or Newey–West when the slopes are autocorrelated. `p.characteristics_panel()` plants a momentum premium of 0.20% a month and gives "size" none.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def fama_macbeth(returns, exposures, nw_lags=0):
    lambdas = []
    for t, t1 in zip(returns.index[:-1], returns.index[1:]):
        X = pd.DataFrame({k: v.loc[t] for k, v in exposures.items()}).dropna()      # characteristics known at t
        y = returns.loc[t1, X.index]
        lambdas.append(sm.OLS(y, sm.add_constant(X)).fit().params)
    L = pd.DataFrame(lambdas)                             # one row of slopes per month
    se = L.std(ddof=1) / np.sqrt(len(L)) if nw_lags == 0 else L.apply(lambda c: p.newey_west_se(c.to_numpy(), nw_lags))
    return pd.DataFrame({"premium": L.mean(), "t_stat": L.mean() / se})

rets, expo = p.characteristics_panel()                    # 100 stocks × 60 months; true momentum premium 0.20%/month
cases = [(rets, expo), (rets, expo, 3)]
mine = [p.attempt(fama_macbeth, *c) for c in cases]
mine = p.check("fama_macbeth", mine, [p.fama_macbeth(*c) for c in cases])
display(mine[0].assign(premium=lambda d: d.premium * 100).rename(columns={"premium": "premium (%/month)"}).round(3))

Momentum comes out at about 0.26% a month (true 0.20%) with t ≈ 3.4; size, which has no premium, is insignificant. Each monthly slope is noisy (100 stocks with 6% idiosyncratic volatility each); only the average over many months is informative. How many months do you need?

In [ ]:
rows = {}
for months in (12, 24, 36, 60):
    fm = p.fama_macbeth(rets.iloc[:months], {k: v.iloc[:months] for k, v in expo.items()})
    rows[f"first {months} months"] = {"momentum premium (%)": fm.loc["momentum", "premium"] * 100,
                                      "momentum t": fm.loc["momentum", "t_stat"], "size t": fm.loc["size", "t_stat"]}
pd.DataFrame(rows).T.round(2)

After 12 months the estimate has the wrong sign; after 24 it happens to look significant, after 36 it doesn't; only the full 60 months give a clear answer. On average the t-statistic grows with √(months), but a short sample can mislead in either direction. A real but small premium needs years of data (which is why the lab also asks whether a premium survives in the second half of the sample).

## 3. Neutralization

A raw signal often hides bets you didn't mean to make: whole sectors, or high-beta stocks. **Neutralize** it: regress the signal across stocks on one dummy per sector (no extra constant) plus beta, and keep the residual. Build the design matrix with `pd.get_dummies(..., dtype=float)` and fit with `sm.OLS(signal, X).fit().resid`.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def neutralize(signal, sectors, beta):
    D = pd.get_dummies(sectors.loc[signal.index], dtype=float)                # one column per sector
    X = pd.concat([D, beta.loc[signal.index].rename("beta")], axis=1)
    return sm.OLS(signal, X).fit().resid

rng = np.random.default_rng(0)
names = [f"N{i:02d}" for i in range(40)]
sectors = pd.Series([f"sec{i % 4}" for i in range(40)], index=names)
beta = pd.Series(rng.normal(1, 0.3, 40), index=names)
signal = pd.Series(rng.normal(0, 1, 40), index=names) + sectors.map({"sec0": 1.0, "sec1": 0.0, "sec2": -0.5, "sec3": 0.3}) + 2 * (beta - 1)
mine = p.check("neutralize", p.attempt(neutralize, signal, sectors, beta), p.neutralize(signal, sectors, beta))
pd.DataFrame({"raw signal": [*signal.groupby(sectors).mean(), np.corrcoef(signal, beta)[0, 1]],
              "neutralized": [*mine.groupby(sectors).mean(), np.corrcoef(mine, beta)[0, 1]]},
             index=[*sorted(sectors.unique()), "correlation with beta"]).round(3)

After neutralization, every sector averages exactly zero and the signal is uncorrelated with beta: a long-short book built on it is sector- and beta-neutral before any optimizer gets involved.

## Wrap-up

* Use Newey–West whenever the observations overlap or persist.
* Fama–MacBeth: one cross-sectional regression per period, then average; premia need long samples.
* Neutralize signals against sectors and beta before trading them.
* Graded version: `labs/part09/week32_statarb` (`newey_west_se`, `fama_macbeth`, `neutralize`).